In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Dict, Tuple, List, Optional
import numpy as np
from dataclasses import dataclass
from torchinfo import summary
import os

# 1. 희범's feature extract base code (CNN + MLP, Total param : 2,531,961)
## 모델 구조 및 채널 변화

### ImageCNN 채널 및 Feature Map Size 변화

| Layer | 입력 | 출력 | 커널 크기 | Feature Map Size | 풀링 |
| --- | --- | --- | --- | --- | --- |
| Input | - | 1채널 | - | (1, 64, 64) | - |
| Conv1 + BN + ReLU | 1채널 | 32채널 | 3×3 | (32, 64, 64) | MaxPool2d(2) |
| After Pool1 | 32채널 | 32채널 | - | (32, 32, 32) | - |
| Conv2 + BN + ReLU | 32채널 | 64채널 | 3×3 | (64, 32, 32) | MaxPool2d(2) |
| After Pool2 | 64채널 | 64채널 | - | (64, 16, 16) | - |
| Conv3 + BN + ReLU | 64채널 | 128채널 | 3×3 | (128, 16, 16) | MaxPool2d(2) |
| After Pool3 | 128채널 | 128채널 | - | (128, 8, 8) | - |
| Conv4 + BN + ReLU | 128채널 | 256채널 | 3×3 | (256, 8, 8) | MaxPool2d(2) |
| After Pool4 | 256채널 | 256채널 | - | (256, 4, 4) | - |
| Flatten | (256, 4, 4) | - | - | 4096 | - |
| FC1 | 4096 | 512 | - | 512 | - |
| Dropout(0.3) | 512 | 512 | - | 512 | - |
| FC_out | 512 | 64 | - | 64 | - |

**채널 변화**: `1 → 32 → 64 → 128 → 256 → 512 → 64`  
**Feature Map Size 변화**: `(1,64,64) → (32,64,64) → (32,32,32) → (64,32,32) → (64,16,16) → (128,16,16) → (128,8,8) → (256,8,8) → (256,4,4) → 4096 → 512 → 64`

### Basic MLP 구조

| Layer | 입력 차원 | 출력 차원 | 활성화 함수 |
| --- | --- | --- | --- |
| Linear | basic_feature_dim | basic_feature_dim × 2 | ReLU |
| BatchNorm1d | basic_feature_dim × 2 | basic_feature_dim × 2 | - |
| Dropout(0.3) | basic_feature_dim × 2 | basic_feature_dim × 2 | - |
| Linear | basic_feature_dim × 2 | 32 | ReLU |

**차원 변화**: `basic_feature_dim → basic_feature_dim × 2 → 32` (예: `41 → 82 → 32`)

### Head (Combined MLP)

| Layer | 입력 차원 | 출력 차원 | 활성화 함수 |
| --- | --- | --- | --- |
| Linear | 96 (64+32) | 64 | ReLU |
| BatchNorm1d | 64 | 64 | - |
| Dropout(0.3) | 64 | 64 | - |
| Linear | 64 | 1 | - |

**차원 변화**: `이미지(64차원) + 기본피처(32차원) → 96차원 → 64차원 → 1차원`

In [27]:
class ImageCNN(nn.Module):
    def __init__(self, output_dim=64, input_size=64):
        super(ImageCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1) 
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        
        self.batch_norm1 = nn.BatchNorm2d(32)
        self.batch_norm2 = nn.BatchNorm2d(64)
        self.batch_norm3 = nn.BatchNorm2d(128)
        self.batch_norm4 = nn.BatchNorm2d(256)
        
        final_size = input_size // 16
        self.fc1 = nn.Linear(256 * final_size * final_size, 512)
        self.fc_out = nn.Linear(512, output_dim) 
        
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.batch_norm1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm2(self.conv2(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm3(self.conv3(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm4(self.conv4(x))), 2)
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc_out(x)


class FullE2EModel(nn.Module):
    def __init__(self, basic_feature_dim, image_cnn_output_dim=64, basic_mlp_output_dim=32, input_grid_size=64):
        super(FullE2EModel, self).__init__()
        
        self.image_cnn = ImageCNN(output_dim=image_cnn_output_dim, input_size=input_grid_size)
        
        self.basic_mlp = nn.Sequential(
            nn.Linear(basic_feature_dim, basic_feature_dim * 2),
            nn.ReLU(),
            nn.BatchNorm1d(basic_feature_dim * 2),
            nn.Dropout(0.3),
            nn.Linear(basic_feature_dim * 2, basic_mlp_output_dim),
            nn.ReLU()
        )
        
        combined_dim = image_cnn_output_dim + basic_mlp_output_dim
        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 1) 
        )
    
    def forward(self, x_image, x_basic):
        img_feat = self.image_cnn(x_image)
        basic_feat = self.basic_mlp(x_basic)
        combined = torch.cat((img_feat, basic_feat), dim=1)
        output = self.head(combined) # 96차원 -> 1차원
        return output

    # 피처 추출을 위한 '머리 없는' forward
    def extract_features(self, x_image, x_basic):
        img_feat = self.image_cnn(x_image)
        basic_feat = self.basic_mlp(x_basic)
        combined = torch.cat((img_feat, basic_feat), dim=1)
        return combined # 96차원 피처 반환

In [28]:
CNN_feature_extractor = FullE2EModel(basic_feature_dim=41)
print(CNN_feature_extractor)

FullE2EModel(
  (image_cnn): ImageCNN(
    (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (batch_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (batch_norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (batch_norm3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (batch_norm4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (fc1): Linear(in_features=4096, out_features=512, bias=True)
    (fc_out): Linear(in_features=512, out_features=64, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (basic_mlp): Sequential(
    (0): Linear(in_features=41, out_fe

In [29]:
p_value_input = torch.randn(1, 1, 64, 64)  # 배치 크기 32, 흑백 이미지 (1 채널), 64x64 크기
p_basic_input = torch.randn(1, 41)  # 배치 크기 32, 기본 피처 41개

dummy_dict_input = {
    'x_image': p_value_input,
    'x_basic': p_basic_input
}

summary(CNN_feature_extractor, input_data=dummy_dict_input, col_names=("input_size", "output_size", "num_params"))

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
FullE2EModel                             --                        [1, 1]                    --
├─ImageCNN: 1-1                          [1, 1, 64, 64]            [1, 64]                   --
│    └─Conv2d: 2-1                       [1, 1, 64, 64]            [1, 32, 64, 64]           320
│    └─BatchNorm2d: 2-2                  [1, 32, 64, 64]           [1, 32, 64, 64]           64
│    └─Conv2d: 2-3                       [1, 32, 32, 32]           [1, 64, 32, 32]           18,496
│    └─BatchNorm2d: 2-4                  [1, 64, 32, 32]           [1, 64, 32, 32]           128
│    └─Conv2d: 2-5                       [1, 64, 16, 16]           [1, 128, 16, 16]          73,856
│    └─BatchNorm2d: 2-6                  [1, 128, 16, 16]          [1, 128, 16, 16]          256
│    └─Conv2d: 2-7                       [1, 128, 8, 8]            [1, 256, 8, 8]            295,168
│    └─BatchNorm2d:

# 2. 극단적으로 간소화된 Feature Extract (Total Param : 553)

## 모델 구조 및 채널 변화

### ImageCNN 채널 및 Feature Map Size 변화

| Layer | 입력 | 출력 | 커널 크기 | Feature Map Size | 풀링 |
| --- | --- | --- | --- | --- | --- |
| Input | - | 1채널 | - | (1, 64, 64) | - |
| Conv1 + BN + ReLU | 1채널 | 8채널 | 3×3 | (8, 64, 64) | MaxPool2d(2) |
| After Pool1 | 8채널 | 8채널 | - | (8, 32, 32) | - |
| AdaptiveAvgPool2d(1) | 8채널 | 8채널 | - | (8, 1, 1) | - |
| Flatten | (8, 1, 1) | - | - | 8 | - |
| FC1 + BN + ReLU | 8 | 8 | - | 8 | - |
| Dropout(0.3) | 8 | 8 | - | 8 | - |

**채널 변화**: `1 → 8 → 8`  
**Feature Map Size 변화**: `(1,64,64) → (8,64,64) → (8,32,32) → (8,1,1) → 8 → 8`

### Basic MLP 구조

| Layer | 입력 차원 | 출력 차원 | 활성화 함수 |
| --- | --- | --- | --- |
| Linear | basic_feature_dim | basic_mlp_output_dim | - |
| BatchNorm1d | basic_mlp_output_dim | basic_mlp_output_dim | - |
| ReLU | basic_mlp_output_dim | basic_mlp_output_dim | ReLU |
| Dropout(0.3) | basic_mlp_output_dim | basic_mlp_output_dim | - |

**차원 변화**: `basic_feature_dim → basic_mlp_output_dim` (예: `41 → 8`)

### Head (Combined MLP)

| Layer | 입력 차원 | 출력 차원 | 활성화 함수 |
| --- | --- | --- | --- |
| Linear | 16 (8+8) | 1 | - |

**차원 변화**: `이미지(8차원) + 기본피처(8차원) → 16차원 → 1차원`


In [30]:
class ImageCNN(nn.Module):
    def __init__(self, output_dim=8, input_size=64):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(2)  # (B,8,32,32)
        )

        self.gap = nn.AdaptiveAvgPool2d(1)  # (B,8,1,1)
        
        self.fc1 = nn.Sequential(
            nn.Linear(8, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        x = self.conv1(x)              # (B,8,32,32)
        x = self.gap(x)                # (B,8,1,1)
        x = x.view(x.size(0), -1)      # (B,8)
        x = self.fc1(x)                # (B,output_dim)
        return x



class FullE2EModel(nn.Module):
    def __init__(self, basic_feature_dim, image_cnn_output_dim=8, basic_mlp_output_dim=8, input_grid_size=64):
        super(FullE2EModel, self).__init__()
        
        self.image_cnn = ImageCNN(output_dim=image_cnn_output_dim, input_size=input_grid_size)
        
        self.basic_mlp = nn.Sequential(
            nn.Linear(basic_feature_dim, basic_mlp_output_dim),
            nn.BatchNorm1d(basic_mlp_output_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        combined_dim = image_cnn_output_dim + basic_mlp_output_dim
        self.head = nn.Sequential(
            nn.Linear(combined_dim, 1)
        )
    
    def forward(self, x_image, x_basic):
        img_feat = self.image_cnn(x_image)
        basic_feat = self.basic_mlp(x_basic)
        combined = torch.cat((img_feat, basic_feat), dim=1)
        output = self.head(combined) # 16차원 -> 1차원
        return output

    # 피처 추출을 위한 '머리 없는' forward
    def extract_features(self, x_image, x_basic):
        img_feat = self.image_cnn(x_image)
        basic_feat = self.basic_mlp(x_basic)
        combined = torch.cat((img_feat, basic_feat), dim=1)
        return combined # 16차원 피처 반환

In [31]:
CNN_feature_extractor = FullE2EModel(basic_feature_dim=41)
print(CNN_feature_extractor)

FullE2EModel(
  (image_cnn): ImageCNN(
    (conv1): Sequential(
      (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (gap): AdaptiveAvgPool2d(output_size=1)
    (fc1): Sequential(
      (0): Linear(in_features=8, out_features=8, bias=True)
      (1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
    )
  )
  (basic_mlp): Sequential(
    (0): Linear(in_features=41, out_features=8, bias=True)
    (1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [32]:
p_value_input = torch.randn(1, 1, 64, 64)  # 배치 크기 32, 흑백 이미지 (1 채널), 64x64 크기
p_basic_input = torch.randn(1, 41)  # 배치 크기 32, 기본 피처 41개

dummy_dict_input = {
    'x_image': p_value_input,
    'x_basic': p_basic_input
}

summary(CNN_feature_extractor, input_data=dummy_dict_input, col_names=("input_size", "output_size", "num_params"))

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
FullE2EModel                             --                        [1, 1]                    --
├─ImageCNN: 1-1                          [1, 1, 64, 64]            [1, 8]                    --
│    └─Sequential: 2-1                   [1, 1, 64, 64]            [1, 8, 32, 32]            --
│    │    └─Conv2d: 3-1                  [1, 1, 64, 64]            [1, 8, 64, 64]            80
│    │    └─BatchNorm2d: 3-2             [1, 8, 64, 64]            [1, 8, 64, 64]            16
│    │    └─ReLU: 3-3                    [1, 8, 64, 64]            [1, 8, 64, 64]            --
│    │    └─MaxPool2d: 3-4               [1, 8, 64, 64]            [1, 8, 32, 32]            --
│    └─AdaptiveAvgPool2d: 2-2            [1, 8, 32, 32]            [1, 8, 1, 1]              --
│    └─Sequential: 2-3                   [1, 8]                    [1, 8]                    --
│    │    └─Linear: 3-5            